# PS3 — Root-Cause Models (all three fleets) → Aurora

Trains a leakage-safe root-cause model per fleet and writes the SQL load file the
dashboard API consumes.

## The leakage problem, and what we predict instead

`component_subsystem` — the target — is supplied by `silver.dim_event_type`
joined on `EVENT_TYPE_ID`. So `event_type_id`, `event_type_name`,
`event_type_desc` and `component_type_name` are **deterministic functions of the
target**. A model given any of them scores ~100% and has learned a join.
That is the PS4 defect verbatim (*"spark_gbt reports test AP 0.9994 … that gap
IS the tell"*).

**What we predict instead:** given this device's *failure history* and the day's
context, which component is failing? *"This device has failed on the card reader
three times in the last 90 days"* is real evidence — available before the event
code is read, and the basis for chaining PS3 onto PS1's three-day-ahead call.

Every history feature is `shift(1)`-ed after sorting, so a row can never see its
own label. Evaluation is a **time split**, not random — device history leaks
across a random split, and a time split is also how it runs in production.

## Read lift and kappa, not accuracy

GATE is 86.6% `DOPP` and VALIDATOR 80.7% `COMMS`. On those fleets an accuracy
near the baseline means the model learned to say the common thing. **Cohen's
kappa near zero says exactly that in one number.** A low lift here is a finding,
not a failure: it would say the diagnostic information lives in the event code
rather than the device's past — which is itself worth telling the client.

## Output

`PS3_sql_outputs/ps3_run_YYYYMMDD.sql` — idempotent INSERTs for
`ps3_model_runs`, `ps3_head_summary`, `ps3_device_predictions`, matching the
deployed Aurora DDL.


In [ ]:
# =============================================================================
# PS3 -- ROOT-CAUSE MODELS, all three fleets, LEAKAGE-SAFE
#
# THE LEAKAGE PROBLEM, stated once. component_subsystem (the target) is supplied
# by silver.dim_event_type joined on EVENT_TYPE_ID. So event_type_id,
# event_type_name, event_type_desc and component_type_name are all deterministic
# functions of the target. A model given any of them scores ~100% and has
# learned a join, not a pattern. That is the PS4 defect verbatim
# ("spark_gbt reports test AP 0.9994 ... that gap IS the tell").
#
# THE HONEST QUESTION INSTEAD: given this device's FAILURE HISTORY and the
# context of the day, which component is failing? That is not circular. "This
# device has failed on the card reader three times in the last 90 days" is real
# evidence, available before the event code is read, and it is what lets PS3
# chain onto PS1's three-day-ahead prediction.
#
# EVALUATION IS TIME-SPLIT, not random. Device history leaks across a random
# split -- the same device appears either side of it. Train on the earlier
# window, test on the later one, which is also how it would run in production.
# =============================================================================
import pandas as pd, numpy as np, json, os, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef,
                             classification_report, confusion_matrix)

OUT = "PS3_spine_outputs"
SUB = {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"}

# Columns that ARE the target, restated so nobody re-adds them later.
LEAKY = ["component_subsystem", "component_type_name", "event_type_name",
         "event_type_desc", "event_type_id", "root_cause", "root_cause_state",
         "sn_root_cause", "sn_severity", "severity_target", "sn_agrees",
         "sn_fault_desc", "sn_label_state", "component_serial_nbr"]

TEST_FRACTION = 0.25          # last 25% of the window, by date
MIN_CLASS_ROWS = 50           # a class the model can never learn is noise


def build_history_features(d):
    """Device failure history as of each device-day, strictly backward-looking.

    Every feature here is computed from events STRICTLY BEFORE the row's own
    transit_day. shift(1) after sorting is what enforces that -- without it the
    row's own label would enter its own features, which is the same leak wearing
    a different hat.
    """
    d = d.sort_values(["device_id", "transit_day"]).copy()
    g = d.groupby("device_id", sort=False)

    d["prior_oos_count"] = g.cumcount()
    d["days_since_prior_oos"] = (
        d["transit_day"] - g["transit_day"].shift(1)).dt.days
    d["device_lifetime_days"] = (
        d["transit_day"] - g["transit_day"].transform("min")).dt.days
    d["prior_oos_rate"] = d["prior_oos_count"] / d["device_lifetime_days"].replace(0, np.nan)

    # the device's PREVIOUS root cause, and how often each cause has occurred
    d["prior_root_cause"] = g["root_cause"].shift(1)
    for cls in d["root_cause"].dropna().unique():
        col = f"prior_share_{cls}"
        hit = (d["root_cause"] == cls).astype(float)
        d[col] = (g[[]].apply(lambda x: x)  # placeholder to keep index alignment
                  if False else
                  hit.groupby(d["device_id"]).apply(
                      lambda s: s.shift(1).expanding().mean()).reset_index(level=0, drop=True))
    return d


def run_fleet(fleet):
    print("=" * 84)
    print(f"PS3 ROOT-CAUSE MODEL -- {fleet}")
    print("=" * 84)
    d = pd.read_parquet(f"{OUT}/{SUB[fleet]}_ps3_spine.parquet")
    d["transit_day"] = pd.to_datetime(d["transit_day"])
    d = d[d["root_cause_state"] == "LABELLED"].copy()
    print(f"  labelled device-days: {len(d):,}")

    keep = d["root_cause"].value_counts()
    keep = keep[keep >= MIN_CLASS_ROWS].index
    dropped = d[~d["root_cause"].isin(keep)]
    if len(dropped):
        print(f"  dropping {len(dropped):,} rows in classes under {MIN_CLASS_ROWS}: "
              f"{sorted(set(dropped['root_cause']))}")
    d = d[d["root_cause"].isin(keep)].copy()

    d = build_history_features(d)

    feat_num = ["prior_oos_count", "days_since_prior_oos", "device_lifetime_days",
                "prior_oos_rate", "hour_of_day", "day_of_week",
                "events_in_day", "informative_in_day", "component_position"]
    feat_num += [c for c in d.columns if c.startswith("prior_share_")]
    feat_cat = [c for c in ("control_group", "transit_mode", "device_type_name",
                            "prior_root_cause") if c in d.columns]
    feat_num = [c for c in feat_num if c in d.columns]

    leaked = [c for c in feat_num + feat_cat if c in LEAKY]
    assert not leaked, f"LEAKAGE: {leaked} would reconstruct the target"

    cut = d["transit_day"].quantile(1 - TEST_FRACTION)
    tr, te = d[d["transit_day"] <= cut], d[d["transit_day"] > cut]
    print(f"  time split at {cut.date()}  train {len(tr):,} | test {len(te):,}")
    if len(te) < 100 or tr["root_cause"].nunique() < 2:
        print("  !! insufficient test data or single class -- skipping")
        return None

    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    Xtr = np.hstack([tr[feat_num].to_numpy(dtype=float),
                     enc.fit_transform(tr[feat_cat].astype(str))]) if feat_cat else tr[feat_num].to_numpy(float)
    Xte = np.hstack([te[feat_num].to_numpy(dtype=float),
                     enc.transform(te[feat_cat].astype(str))]) if feat_cat else te[feat_num].to_numpy(float)
    ytr, yte = tr["root_cause"].to_numpy(), te["root_cause"].to_numpy()

    t0 = time.time()
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                         max_depth=6, random_state=42,
                                         categorical_features=None)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    secs = time.time() - t0

    base_cls = pd.Series(ytr).value_counts().index[0]
    base_acc = (yte == base_cls).mean()
    acc  = accuracy_score(yte, pred)
    f1m  = f1_score(yte, pred, average="macro", zero_division=0)
    f1w  = f1_score(yte, pred, average="weighted", zero_division=0)
    bal  = balanced_accuracy_score(yte, pred)
    kap  = cohen_kappa_score(yte, pred)
    mcc  = matthews_corrcoef(yte, pred)

    print(f"\n  trained in {secs:.0f}s | {len(feat_num)+len(feat_cat)} features, "
          f"{len(set(ytr))} classes")
    print(f"  {'accuracy':22s} {acc:.4f}")
    print(f"  {'majority baseline':22s} {base_acc:.4f}   (always predict {base_cls})")
    print(f"  {'LIFT over baseline':22s} {acc-base_acc:+.4f}   <- the number that matters")
    print(f"  {'macro F1':22s} {f1m:.4f}")
    print(f"  {'weighted F1':22s} {f1w:.4f}")
    print(f"  {'balanced accuracy':22s} {bal:.4f}")
    print(f"  {'Cohen kappa':22s} {kap:.4f}   (0 = no better than chance)")
    print(f"  {'MCC':22s} {mcc:.4f}")
    if acc - base_acc < 0.02:
        print("\n  !! NO MEANINGFUL LIFT. Device history does not predict which")
        print("     component fails. The diagnostic information lives in the event")
        print("     code, not in the device's past -- which is itself the finding.")
    print("\n  per class:")
    print(classification_report(yte, pred, zero_division=0))

    return {"fleet": fleet, "n_train": int(len(tr)), "n_test": int(len(te)),
            "n_features": int(len(feat_num) + len(feat_cat)),
            "n_classes": int(len(set(ytr))),
            "class_labels": ",".join(sorted(set(ytr))),
            "accuracy": round(float(acc), 4), "baseline": round(float(base_acc), 4),
            "lift": round(float(acc - base_acc), 4),
            "f1_macro": round(float(f1m), 4), "f1_weighted": round(float(f1w), 4),
            "balanced_accuracy": round(float(bal), 4),
            "cohen_kappa": round(float(kap), 4), "mcc": round(float(mcc), 4),
            "champion": "HistGradientBoosting", "split_date": str(cut.date()),
            "predictions": te.assign(pred_root_cause=pred)}


results = {}
for fleet in SUB:
    r = run_fleet(fleet)
    if r: results[fleet] = r
    print()

print("=" * 84)
print("PS3 ROOT-CAUSE MODEL SUMMARY -- all fleets")
print("=" * 84)
print(f"  {'fleet':10s} {'classes':>8s} {'accuracy':>9s} {'baseline':>9s} {'lift':>8s} "
      f"{'macroF1':>8s} {'kappa':>7s}")
for f, r in results.items():
    print(f"  {f:10s} {r['n_classes']:8d} {r['accuracy']:9.4f} {r['baseline']:9.4f} "
          f"{r['lift']:+8.4f} {r['f1_macro']:8.4f} {r['cohen_kappa']:7.4f}")
print("""
  READ LIFT AND KAPPA, NOT ACCURACY. On GATE and VALIDATOR one class is ~80-87%
  of the data, so accuracy near the baseline means the model has learned to say
  the common thing. Cohen's kappa near 0 says the same in one number.""")


In [ ]:
# =============================================================================
# PS3 -- EXPORT TO AURORA  (dashboard load file)
#
# The dashboard API loads PS3 from SQL files under sql/load/, not parquet:
#     aws lambda invoke ... --payload '{"action":"load_run","only":["ps3"]}'
# So this writes INSERT statements matching the live Aurora DDL exactly.
#
# Tables written:
#   ps3_model_runs           one row  -- run provenance
#   ps3_head_summary         one row per fleet -- metrics the dashboard renders
#   ps3_device_predictions   one row per device -- the drill-down
#
# Column names and types are taken from the deployed DDL in
# sql/15_phase2a_ps3_two_head.sql. If a load fails on a column, that DDL is the
# source of truth, not this file.
# =============================================================================
import pandas as pd, numpy as np, datetime as dt, os

CITY        = "CHI"
AS_OF       = "2026-04-11"                     # matches the spine RUN_DATE
RUN_ID      = f"ps3_oos_{dt.datetime.now().strftime('%Y%m%d')}"
NOTEBOOK    = "PS3_03_RootCause_Models.ipynb"
MACRO_F1_FLOOR = 0.45                          # the existing PS3 root_cause gate

def q(v):
    """SQL literal. None/NaN -> NULL, strings escaped."""
    if v is None or (isinstance(v, float) and np.isnan(v)): return "NULL"
    if isinstance(v, (bool, np.bool_)):  return "TRUE" if v else "FALSE"
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return f"{float(v):.6f}"
    return "'" + str(v).replace("'", "''") + "'"

lines = [
    "-- =========================================================================",
    f"-- PS3 root-cause run {RUN_ID}",
    f"-- generated {dt.datetime.now().isoformat(timespec='seconds')}",
    "--",
    "-- Event contract: docs/OOS_EVENT_CONTRACT.md (2026-08-03.v5)",
    "--   silver.device_event_enriched | is_hardware_oos_event=TRUE",
    "--   | UPPER(TRIM(EVENT_STATE_TYPE_NAME))='SET' | calendar-day grain",
    "--",
    "-- Supersedes ps3_20260726, which sat on silver.incident_root_cause",
    "-- (ServiceNow availability events, the CHARGEABLE subset) and therefore",
    "-- had 0 validator rows by construction.",
    "-- =========================================================================",
    "BEGIN;",
    "",
    f"DELETE FROM ps3_device_predictions WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    f"DELETE FROM ps3_head_summary       WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    f"DELETE FROM ps3_model_runs         WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    "",
]

# --- ps3_model_runs ----------------------------------------------------------
n_inc = int(sum(r["n_train"] + r["n_test"] for r in results.values()))
lines += [
    "-- run provenance -----------------------------------------------------------",
    "INSERT INTO ps3_model_runs (city_id, run_id, run_ts, run_kind, source_notebook,",
    "  n_incidents, device_scope, mlflow_version, severity_collapse_verified, notes, as_of_date)",
    "VALUES (" + ", ".join([
        q(CITY), q(RUN_ID), q(dt.datetime.now().isoformat(timespec='seconds')),
        q("train"), q(NOTEBOOK), q(n_inc), q(",".join(results.keys())), q("v3"),
        q(False),
        q("Rebuilt on the hardware-OOS contract. Root cause = device-native "
          "component_subsystem. Event-type columns EXCLUDED as leakage. "
          "Time-split evaluation."),
        q(AS_OF)]) + ")",
    "ON CONFLICT (city_id, run_id) DO NOTHING;",
    "",
]

# --- ps3_head_summary --------------------------------------------------------
lines += ["-- per-fleet metrics --------------------------------------------------------"]
for fleet, r in results.items():
    gate_pass = bool(r["f1_macro"] >= MACRO_F1_FLOOR)
    lines += [
        "INSERT INTO ps3_head_summary (city_id, run_id, device_category, head, modeled,",
        "  target_col, champion, n_classes, class_labels, test_f1_macro, test_f1_weighted,",
        "  test_accuracy, test_balanced_accuracy, test_cohen_kappa, test_mcc,",
        "  macro_f1_floor, gate_pass, n_train, n_test, n_features, rootcause_source, as_of_date)",
        "VALUES (" + ", ".join([
            q(CITY), q(RUN_ID), q(fleet), q("root_cause"), q(True),
            q("component_subsystem"), q(r["champion"]), q(r["n_classes"]),
            q(r["class_labels"]), q(r["f1_macro"]), q(r["f1_weighted"]),
            q(r["accuracy"]), q(r["balanced_accuracy"]), q(r["cohen_kappa"]),
            q(r["mcc"]), q(MACRO_F1_FLOOR), q(gate_pass),
            q(r["n_train"]), q(r["n_test"]), q(r["n_features"]),
            q("device_event_enriched.component_subsystem"), q(AS_OF)]) + ")",
        "ON CONFLICT (city_id, run_id, device_category, head) DO NOTHING;",
        "",
    ]

# --- ps3_device_predictions --------------------------------------------------
lines += ["-- per-device drill-down ----------------------------------------------------"]
rowct = 0
for fleet, r in results.items():
    p = r["predictions"]
    agg = (p.groupby("device_id")
             .agg(n_incidents=("device_id", "size"),
                  dominant_pred_component=("pred_root_cause",
                                           lambda s: s.value_counts().index[0]),
                  last_incident_dtm=("event_dtm", "max"))
             .reset_index())
    for _, row in agg.iterrows():
        lines.append(
            "INSERT INTO ps3_device_predictions (city_id, run_id, device_id, "
            "mars_device_category, n_incidents, dominant_pred_component, "
            "last_incident_dtm, computed_date) VALUES ("
            + ", ".join([q(CITY), q(RUN_ID), q(row["device_id"]), q(fleet),
                         q(int(row["n_incidents"])), q(row["dominant_pred_component"]),
                         q(str(row["last_incident_dtm"])), q(AS_OF)])
            + ") ON CONFLICT (city_id, run_id, device_id) DO NOTHING;")
        rowct += 1

lines += ["", "COMMIT;", ""]

os.makedirs("PS3_sql_outputs", exist_ok=True)
path = f"PS3_sql_outputs/ps3_run_{dt.datetime.now().strftime('%Y%m%d')}.sql"
with open(path, "w") as fh:
    fh.write("\n".join(lines))

print("=" * 84)
print("SQL LOAD FILE WRITTEN")
print("=" * 84)
print(f"  {path}")
print(f"  {os.path.getsize(path):,} bytes")
print(f"  ps3_model_runs         1 row")
print(f"  ps3_head_summary       {len(results)} rows")
print(f"  ps3_device_predictions {rowct:,} rows")
print(f"\n  run_id: {RUN_ID}")
print(f"""
  TO LOAD IT
    1. copy this file into the repo at
         api/lambda/cubic-mars-dashboard-api/sql/load/
    2. add it to the LOAD_FILES list in handler.py next to the ps3 entry
    3. redeploy the dashboard-api and invoke:
         aws lambda invoke --function-name cubic-mars-dashboard-api \\
           --cli-binary-format raw-in-base64-out \\
           --payload '{{"action":"load_run","only":["ps3"]}}' out.json

  The DELETEs at the top make it idempotent -- re-running replaces this run_id
  rather than duplicating it.""")
print("=" * 84)


## Getting it onto the dashboard

1. Copy `PS3_sql_outputs/ps3_run_*.sql` into
   `api/lambda/cubic-mars-dashboard-api/sql/load/`
2. Register it in `handler.py` beside the existing `("ps3", "sql/load/ps3_run_20260726.sql")`
3. Redeploy dashboard-api, then:

```bash
AWS_MAX_ATTEMPTS=1 aws lambda invoke --function-name cubic-mars-dashboard-api \
  --region us-east-1 --cli-binary-format raw-in-base64-out \
  --payload '{"action":"load_run","only":["ps3"]}' /tmp/ps3load.json
python3 -m json.tool /tmp/ps3load.json
```

The `DELETE`s at the top of the file make it idempotent — re-running replaces
that `run_id` rather than duplicating it.

## What this run supersedes

`ps3_20260726` sat on `silver.incident_root_cause` — ServiceNow availability
events, the chargeable subset — and had **zero validator rows by construction**.
This run is on the hardware-OOS contract and covers all three fleets.
